In [45]:
%load_ext autoreload
%autoreload 2
from form4_cache_manager import (
    DATA_DIR,
    load_or_update_form4_for_tickers,
    load_or_update_form4_for_ticker,
)
from price_cache_manager import load_or_update_prices
import pandas as pd
from backtest_engine import run_backtest

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
tickers = ["AAPL", "META", "GOOGL", "AMZN"]

filings_all, common_all, options_all = load_or_update_form4_for_tickers(
    tickers,
    n_if_no_cache=None,   # 第一次建议先用 50，稳定后再改 None
    data_dir=DATA_DIR,
)


Loading/updating Form 4 data for AAPL...
Cache found for AAPL. Checking for updates...
AAPL: cache is already up to date.
Finished AAPL. Sleeping 6.0s before next ticker...

Loading/updating Form 4 data for META...
Cache found for META. Checking for updates...
META: cache is already up to date.
Finished META. Sleeping 6.7s before next ticker...


In [ ]:
tickers = ["AAPL", "NVDA", "MSFT", "AMD"]

df_all = load_or_update_prices(tickers)


Processing AAPL...
AAPL: cache found, checking updates...
AAPL: last cached date = 2026-04-24 00:00:00
AAPL: found 1 new rows
Saved cache for AAPL → price_data/AAPL_ohlcv.csv

Processing NVDA...
NVDA: cache found, checking updates...
NVDA: last cached date = 2026-04-24 00:00:00
NVDA: found 1 new rows
Saved cache for NVDA → price_data/NVDA_ohlcv.csv

Processing MSFT...
MSFT: cache found, checking updates...
MSFT: last cached date = 2026-04-24 00:00:00
MSFT: found 1 new rows
Saved cache for MSFT → price_data/MSFT_ohlcv.csv

Processing AMD...
AMD: cache found, checking updates...
AMD: last cached date = 2026-04-24 00:00:00
AMD: found 1 new rows
Saved cache for AMD → price_data/AMD_ohlcv.csv

Final shape: (92, 34)


In [ ]:
common_all.columns

Index(['issuer_cik', 'issuer_ticker', 'owner_name', 'is_director',
       'is_officer', 'officer_title', 'is_ten_percent_owner', 'security_title',
       'transaction_date', 'transaction_code', 'shares', 'price_per_share',
       'acquired_disposed_code', 'shares_owned_following', 'ownership_nature',
       'ticker', 'filing_date', 'acceptance_datetime', 'accession_number',
       'xml_url'],
      dtype='str')

In [ ]:
#clean the data
df = common_all.copy()

df["transaction_date"] = pd.to_datetime(df["transaction_date"])
df["shares"] = pd.to_numeric(df["shares"], errors="coerce")
df["price_per_share"] = pd.to_numeric(df["price_per_share"], errors="coerce")

df["signed_shares"] = df["shares"]

# 买 + 卖 -
df.loc[df["acquired_disposed_code"] == "D", "signed_shares"] *= -1

In [ ]:
# 因子1：净买入量
factor1 = (
    df.groupby(["ticker", "transaction_date"])["signed_shares"]
    .sum()
    .rename("insider_net_shares")
)

In [ ]:
# 因子2：买卖次数 imbalance
df["is_buy"] = (df["acquired_disposed_code"] == "A").astype(int)
df["is_sell"] = (df["acquired_disposed_code"] == "D").astype(int)

factor2 = (
    df.groupby(["ticker", "transaction_date"])[["is_buy", "is_sell"]]
    .sum()
)

factor2["buy_ratio"] = factor2["is_buy"] / (factor2["is_buy"] + factor2["is_sell"] + 1e-6)

In [ ]:
def str_to_float_bool(s):
    return s.astype(str).str.lower().map({
        "true": 1.0,
        "false": 0.0
    }).fillna(0.0)
df["importance"] = (
    str_to_float_bool(df["is_officer"])
    + str_to_float_bool(df["is_director"])
    + str_to_float_bool(df["is_ten_percent_owner"])
)

In [ ]:
# 因子3：insider重要性加权
factor3 = (
    df.groupby(["ticker", "transaction_date"])
    .apply(lambda x: (x["signed_shares"] * x["importance"]).sum())
    .rename("weighted_insider_flow")
)

In [ ]:
factors = pd.concat([factor1, factor2, factor3], axis=1).reset_index()

In [ ]:
factors = factors.rename(columns={
    "transaction_date": "trade_date"
})

df_all = df_all.rename(columns={
    "date": "trade_date"
})

In [ ]:
factors.columns

Index(['ticker', 'trade_date', 'insider_net_shares', 'is_buy', 'is_sell',
       'buy_ratio', 'weighted_insider_flow'],
      dtype='str')

In [ ]:
print(df_all.columns)

Index([               'date',           'adj_close',               'close',
                      'high',                 'low',                'open',
                    'volume',              'ticker',          ('date', ''),
       ('adj_close', 'AAPL'),     ('close', 'AAPL'),      ('high', 'AAPL'),
             ('low', 'AAPL'),      ('open', 'AAPL'),    ('volume', 'AAPL'),
              ('ticker', ''), ('adj_close', 'NVDA'),     ('close', 'NVDA'),
            ('high', 'NVDA'),       ('low', 'NVDA'),      ('open', 'NVDA'),
          ('volume', 'NVDA'), ('adj_close', 'MSFT'),     ('close', 'MSFT'),
            ('high', 'MSFT'),       ('low', 'MSFT'),      ('open', 'MSFT'),
          ('volume', 'MSFT'),  ('adj_close', 'AMD'),      ('close', 'AMD'),
             ('high', 'AMD'),        ('low', 'AMD'),       ('open', 'AMD'),
           ('volume', 'AMD')],
      dtype='object')


In [ ]:
df_all.head(10)

,date,adj_close,close,high,low,open,volume,ticker,"(date, )","(adj_close, AAPL)",...,"(high, MSFT)","(low, MSFT)","(open, MSFT)","(volume, MSFT)","(adj_close, AMD)","(close, AMD)","(high, AMD)","(low, AMD)","(open, AMD)","(volume, AMD)"
0,2026-03-25,252.6199951171875,252.6199951171875,255.0,251.60000610351562,254.10000610351562,28476700,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-26,252.88999938964844,252.88999938964844,257.0,250.77000427246094,252.1199951171875,41796700,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-27,248.8000030517578,248.8000030517578,255.49000549316406,248.07000732421875,253.89999389648438,47900000,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-30,246.6300048828125,246.6300048828125,250.8699951171875,245.50999450683594,250.07000732421875,39446200,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-31,253.7899932861328,253.7899932861328,255.47999572753906,247.10000610351562,247.91000366210938,49598100,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-04-01,255.6300048828125,255.6300048828125,256.17999267578125,253.3300018310547,254.0800018310547,40059400,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-04-02,255.9199981689453,255.9199981689453,256.1300048828125,250.64999389648438,254.1999969482422,31289400,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-04-06,258.8599853515625,258.8599853515625,262.1600036621094,256.4599914550781,256.510009765625,29329900,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-04-07,253.5,253.5,256.20001220703125,245.6999969482422,256.1600036621094,62148000,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-04-08,258.8999938964844,258.8999938964844,259.75,256.5299987792969,258.45001220703125,41032800,AAPL,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
res1 = run_backtest(
    factor_df=factors[["trade_date", "ticker", "insider_net_shares"]],
    price_df=df_all,
    factor_col="insider_net_shares"
)

KeyError: 'trade_date'

In [ ]:
res2 = run_backtest(
    factor_df=factors[["trade_date", "ticker", "buy_ratio"]],
    price_df=price,
    factor_col="buy_ratio"
)

In [ ]:
res3 = run_backtest(
    factor_df=factors[["trade_date", "ticker", "weighted_insider_flow"]],
    price_df=price,
    factor_col="weighted_insider_flow"
)